In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
DB_PATH = Path.home() / ".aivas" / "aivas.db"
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql("SELECT * FROM cves WHERE cvss_score IS NOT NULL", conn)
print(f"Loaded {len(df):,} CVEs with CVSS scores")
df.head()


## CVE Count by Year


In [ ]:
df["year"] = pd.to_datetime(df["published"]).dt.year
yearly = df.groupby("year").size().reset_index(name="count")

plt.figure(figsize=(12, 4))
sns.barplot(data=yearly, x="year", y="count", color="steelblue")
plt.title("CVEs Published per Year")
plt.xlabel("Year"); plt.ylabel("CVE Count")
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig("notebooks/yearly_cves.png", dpi=150); plt.show()


## Severity Distribution


In [ ]:
sev_order = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
sev_counts = df["cvss_severity"].value_counts().reindex(sev_order, fill_value=0)
colors = ["#d32f2f", "#f57c00", "#fbc02d", "#388e3c"]
plt.figure(figsize=(6, 4))
sev_counts.plot(kind="bar", color=colors)
plt.title("CVE Severity Distribution"); plt.xlabel("Severity"); plt.ylabel("Count")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig("notebooks/severity_distribution.png", dpi=150); plt.show()
print(sev_counts.to_string())


## CVSS Score Histogram


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["cvss_score"], bins=30, kde=True, color="steelblue")
plt.title("CVSS Score Distribution"); plt.xlabel("CVSS Score"); plt.ylabel("Frequency")
plt.tight_layout(); plt.savefig("notebooks/cvss_histogram.png", dpi=150); plt.show()
print(df["cvss_score"].describe().round(2))


## Attack Vector Breakdown


In [ ]:
av = df["attack_vector"].value_counts()
plt.figure(figsize=(6, 6))
av.plot(kind="pie", autopct="%1.1f%%", startangle=90,
        colors=sns.color_palette("pastel"))
plt.title("CVE Attack Vector Breakdown"); plt.ylabel("")
plt.tight_layout(); plt.savefig("notebooks/attack_vector_pie.png", dpi=150); plt.show()


## Top 10 Vulnerable Products


In [ ]:
cpe_df = pd.read_sql(
    "SELECT cpe_criteria, COUNT(DISTINCT cve_id) as cve_count"
    " FROM cpe_matches WHERE vulnerable = 1"
    " GROUP BY cpe_criteria ORDER BY cve_count DESC LIMIT 10",
    conn,
)
cpe_df["label"] = cpe_df["cpe_criteria"].str.split(":").str[4]
plt.figure(figsize=(10, 4))
sns.barplot(data=cpe_df, x="cve_count", y="label", orient="h", color="coral")
plt.title("Top 10 Most Vulnerable Products (by CVE Count)")
plt.xlabel("Distinct CVE Count"); plt.ylabel("Product")
plt.tight_layout(); plt.savefig("notebooks/top_vulnerable_products.png", dpi=150); plt.show()


## Top 10 CWE Types


In [ ]:
cwe_df = df[df["cwe_id"].notna()].copy()
top_cwe = cwe_df["cwe_id"].value_counts().head(10).reset_index()
top_cwe.columns = ["cwe_id", "count"]
plt.figure(figsize=(10, 4))
sns.barplot(data=top_cwe, x="count", y="cwe_id", orient="h", color="mediumpurple")
plt.title("Top 10 Most Common Weakness Types (CWE)")
plt.xlabel("CVE Count"); plt.ylabel("CWE ID")
plt.tight_layout(); plt.savefig("notebooks/top_cwe.png", dpi=150); plt.show()


## Summary Statistics


In [ ]:
summary = {
    "Total CVEs (scored)": len(df),
    "Critical CVEs": int((df["cvss_severity"] == "CRITICAL").sum()),
    "High CVEs": int((df["cvss_severity"] == "HIGH").sum()),
    "Network-exploitable CVEs": int((df["attack_vector"] == "NETWORK").sum()),
    "Mean CVSS Score": round(df["cvss_score"].mean(), 2),
    "Median CVSS Score": round(df["cvss_score"].median(), 2),
    "Years covered": f"{int(df['year'].min())} - {int(df['year'].max())}",
}
for k, v in summary.items():
    print(f"{k:<35} {v}")
conn.close()
